# 07 — Current client value score (deterministic)

Computes each client’s **current** value contribution to the bank using only available current-state signals (balances, transaction activity, product depth, primary-income relationship, credit quality, and tenure).

This notebook **does not train or use a regression model** and does not predict future value.

## Outputs
- `data/processed/features_value.parquet`
- `data/models/value_weights.json`

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import json
from statistics import NormalDist

import numpy as np
import pandas as pd

import config

PROC = config.PROCESSED_DATA_DIR
MODELS = config.MODELS_DIR
MODELS.mkdir(parents=True, exist_ok=True)

## 1) Load inputs

We merge engineered features with raw client attributes needed for credit-rating convention and tenure derivation.

In [3]:
features = pd.read_parquet(PROC / 'features_churn.parquet')
clients = pd.read_parquet(PROC / 'clients_eligible.parquet')
with open(PROC / 'time_split.json') as f:
    time_split = json.load(f)

reference_date = pd.Timestamp(time_split['feature_window_end'])

df = features.merge(
    clients[['IDENTIFIKATOR_KLIJENTA', 'KREDITNI_RATING', 'DATUM_PRVOG_POCETKA_POSLOVNOG_ODNOSA']],
    on='IDENTIFIKATOR_KLIJENTA',
    how='inner',
)

assert df['IDENTIFIKATOR_KLIJENTA'].is_unique, 'Duplicate client IDs after merge.'
print(f'features shape : {features.shape}')
print(f'clients shape  : {clients.shape}')
print(f'merged shape   : {df.shape}')
print(f'reference_date : {reference_date}')

features shape : (4614, 83)
clients shape  : (4614, 39)
merged shape   : (4614, 85)
reference_date : 2025-12-15 23:55:04+00:00


## 2) Deterministic component scores (current value drivers)

All components are deterministic and normalized to [0, 1] via percentile ranks so they are comparable in a weighted sum.

In [4]:
normal_dist = NormalDist()

def pct_rank(series: pd.Series) -> pd.Series:
    return series.rank(pct=True, method='average', na_option='bottom').astype(float)

def as_numeric(series: pd.Series, fill_value: float = 0.0) -> pd.Series:
    return pd.to_numeric(series, errors='coerce').fillna(fill_value)

# 2.1 Balance contribution (funding proxy): only non-negative balances contribute positively
balance_nonnegative = as_numeric(df['total_balance_avg'], fill_value=0.0).clip(lower=0.0)
df['balance_score'] = pct_rank(np.log1p(balance_nonnegative))

# 2.2 Revenue/activity contribution: combine txn amount and txn count
tx_amount_nonnegative = as_numeric(df['tx_amount_total'], fill_value=0.0).clip(lower=0.0)
tx_count_nonnegative = as_numeric(df['tx_count_total'], fill_value=0.0).clip(lower=0.0)
tx_amount_rank = pct_rank(np.log1p(tx_amount_nonnegative))
tx_count_rank = pct_rank(np.log1p(tx_count_nonnegative))
df['revenue_score'] = pct_rank(0.75 * tx_amount_rank + 0.25 * tx_count_rank)

# 2.3 Product-depth contribution
has_mortgage = as_numeric(df['has_mortgage'])
has_loan = as_numeric(df['has_loan'])
has_investment = as_numeric(df['has_investment'])
has_credit_card = as_numeric(df['has_credit_card'])
has_checking = as_numeric(df['has_checking'])
has_savings = as_numeric(df['has_savings'])

df['product_depth_raw'] = (
    3.0 * (has_mortgage + has_loan + has_investment)
    + 2.0 * has_credit_card
    + 1.0 * (has_checking + has_savings)
)
df['product_depth_score'] = pct_rank(df['product_depth_raw'])

# 2.4 Primary-income relationship contribution
df['primary_bank_score'] = as_numeric(df['receives_primary_income_at_bank']).clip(lower=0.0, upper=1.0)

# 2.5 Credit quality contribution (lower raw rating number = better)
cr = pd.to_numeric(df['KREDITNI_RATING'], errors='coerce')
if cr.notna().any():
    cr_imputed = cr.fillna(cr.median())
    df['credit_rating_score'] = pct_rank(-cr_imputed)
else:
    df['credit_rating_score'] = 0.5

# 2.6 Tenure contribution (derived from raw client relationship start date)
first_relationship_date = pd.to_datetime(df['DATUM_PRVOG_POCETKA_POSLOVNOG_ODNOSA'], utc=True, errors='coerce')
tenure_months = ((reference_date - first_relationship_date).dt.total_seconds() / (30.4375 * 24 * 3600)).clip(lower=0.0)
if tenure_months.notna().any():
    tenure_months_filled = tenure_months.fillna(tenure_months.median())
    df['tenure_months'] = tenure_months_filled
    df['tenure_score'] = pct_rank(tenure_months_filled)
else:
    df['tenure_months'] = 0.0
    df['tenure_score'] = 0.5

component_cols = list(config.VALUE_WEIGHTS.keys())
print('Component summary:')
print(df[component_cols].describe().T[['min', '25%', '50%', '75%', 'max', 'mean']].round(4))

Component summary:
                        min     25%     50%     75%     max    mean
balance_score        0.1939  0.1939  0.5001  0.7501  1.0000  0.5001
revenue_score        0.0002  0.2502  0.5001  0.7501  1.0000  0.5001
product_depth_score  0.1467  0.1467  0.4598  0.7607  0.9985  0.5001
tenure_score         0.0002  0.2496  0.5002  0.7500  0.9997  0.5001
primary_bank_score   0.0000  0.0000  0.0000  1.0000  1.0000  0.4469
credit_rating_score  0.0009  0.5354  0.5354  0.5354  0.9998  0.5001


## 3) Weighted score + normal-like calibration

1. Compute a weighted deterministic base score (`value_score_raw`).
2. Convert to percentile rank across all eligible clients.
3. Apply inverse normal CDF for a bell-shaped calibration axis (`value_score_z`).
4. Linearly map z in [-3, 3] to final [0, 1] `value_score`.

In [5]:
assert set(component_cols) == set(config.VALUE_WEIGHTS.keys()), 'Component/weight mismatch.'

df['value_score_raw'] = sum(df[c] * w for c, w in config.VALUE_WEIGHTS.items()).clip(0.0, 1.0)
df['value_score_percentile'] = pct_rank(df['value_score_raw']).clip(1e-4, 1 - 1e-4)
df['value_score_z'] = df['value_score_percentile'].map(normal_dist.inv_cdf)
df['value_score'] = ((df['value_score_z'] + 3.0) / 6.0).clip(0.0, 1.0)

tier_labels = list(config.VALUE_TIER_LABELS)
tier_z_bins = list(config.VALUE_TIER_Z_BINS)
df['value_tier'] = pd.cut(
    df['value_score_z'],
    bins=tier_z_bins,
    labels=tier_labels,
    include_lowest=True,
).astype(str)

print('value_score_raw summary:')
print(df['value_score_raw'].describe().round(4).to_string())
print('\nvalue_score (calibrated) summary:')
print(df['value_score'].describe().round(4).to_string())
print('\nvalue_score_z summary:')
print(df['value_score_z'].describe().round(4).to_string())
print('\nTier distribution:')
print(df['value_tier'].value_counts().reindex(tier_labels).to_string())

value_score_raw summary:
count    4614.0000
mean        0.4948
std         0.2215
min         0.1173
25%         0.2794
50%         0.5044
75%         0.6750
max         0.9843

value_score (calibrated) summary:
count    4614.0000
mean        0.5001
std         0.1663
min         0.0000
25%         0.3877
50%         0.5000
75%         0.6124
max         1.0000

value_score_z summary:
count    4614.0000
mean        0.0008
std         0.9999
min        -3.5188
25%        -0.6740
50%         0.0003
75%         0.6747
max         3.7190

Tier distribution:
value_tier
bronze       732
silver      1575
gold        1845
platinum     462


## 4) Save outputs

In [6]:
diagnostic_cols = [
    'product_depth_raw',
    'tenure_months',
    'value_score_raw',
    'value_score_percentile',
    'value_score_z',
]

out_cols = ['IDENTIFIKATOR_KLIJENTA'] + component_cols + ['value_score', 'value_tier'] + diagnostic_cols
out_df = df[out_cols].copy()

out_parquet = PROC / 'features_value.parquet'
out_df.to_parquet(out_parquet, index=False)

out_weights = MODELS / 'value_weights.json'
serialized_z_bins = [
    '-inf' if z == -float('inf') else 'inf' if z == float('inf') else float(z)
    for z in tier_z_bins
]
payload = {
    'weights': config.VALUE_WEIGHTS,
    'tiering': {
        'method': 'zscore_bins_from_percentile_calibration',
        'z_bins': serialized_z_bins,
        'labels': tier_labels,
        'score_mapping': 'value_score = clip((value_score_z + 3) / 6, 0, 1)',
    },
    'diagnostic_columns': diagnostic_cols,
    'notes': {
        'approach': 'Deterministic current-value scoring. No fitted model and no future prediction target.',
        'credit_rating_convention': 'Lower numeric KREDITNI_RATING is better; component is percentile rank of -rating.',
        'tenure_definition': 'Months from DATUM_PRVOG_POCETKA_POSLOVNOG_ODNOSA to feature_window_end.',
        'normal_calibration': 'Raw weighted score -> percentile -> inverse normal CDF -> linear map to [0,1].',
    },
}
with open(out_weights, 'w') as f:
    json.dump(payload, f, indent=2)

print(f'Saved: {out_parquet}   shape={out_df.shape}')
print(f'Saved: {out_weights}')

Saved: /Users/Max/Desktop/Fintech-hackathon/data/processed/features_value.parquet   shape=(4614, 14)
Saved: /Users/Max/Desktop/Fintech-hackathon/data/models/value_weights.json


## 5) Sanity checks

In [7]:
check = pd.read_parquet(PROC / 'features_value.parquet')
expected_rows = len(clients)

assert len(check) == expected_rows, f'Row count mismatch: {len(check)} != {expected_rows}'
assert check['IDENTIFIKATOR_KLIJENTA'].is_unique, 'Duplicate client IDs in output.'

for c in component_cols:
    assert check[c].between(0.0, 1.0).all(), f'{c} outside [0,1]'

assert check['value_score'].between(0.0, 1.0).all(), 'value_score outside [0,1]'
assert check['value_score_raw'].between(0.0, 1.0).all(), 'value_score_raw outside [0,1]'
assert check['value_score_percentile'].between(0.0, 1.0).all(), 'value_score_percentile outside [0,1]'
assert np.isfinite(check['value_score_z']).all(), 'value_score_z contains non-finite values'

valid_tiers = {'bronze', 'silver', 'gold', 'platinum'}
assert set(check['value_tier'].unique()).issubset(valid_tiers), 'Unexpected tier values detected'

nan_cols = [c for c in check.columns if check[c].isna().any()]
assert not nan_cols, f'NaNs present in output columns: {nan_cols}'

print('All sanity checks passed.')
print('\nTier distribution:')
print(check['value_tier'].value_counts().reindex(['bronze', 'silver', 'gold', 'platinum']).to_string())
print('\nCalibrated score summary:')
print(check['value_score'].describe().round(4).to_string())
print('\nZ calibration summary:')
print(check['value_score_z'].describe().round(4).to_string())

All sanity checks passed.

Tier distribution:
value_tier
bronze       732
silver      1575
gold        1845
platinum     462

Calibrated score summary:
count    4614.0000
mean        0.5001
std         0.1663
min         0.0000
25%         0.3877
50%         0.5000
75%         0.6124
max         1.0000

Z calibration summary:
count    4614.0000
mean        0.0008
std         0.9999
min        -3.5188
25%        -0.6740
50%         0.0003
75%         0.6747
max         3.7190
